<a href="https://colab.research.google.com/github/evanssignvrse/mediapipe-Pose_Landmarker/blob/main/holistic%20landmarks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install MediaPipe
!pip install -q mediapipe

# Download the model bundles
!wget -O face_landmarker.task -q https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task
!wget -O pose_landmarker.task -q https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_heavy/float16/1/pose_landmarker_heavy.task
!wget -O hand_landmarker.task -q https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 10.4 MB/s eta 0:00:00


In [5]:
import cv2
import mediapipe as mp
import numpy as np
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from mediapipe.tasks.python.vision import drawing_utils
from mediapipe.tasks.python.vision import drawing_styles

def draw_all_landmarks(rgb_image, face_result, pose_result, hand_result):
    annotated_image = np.copy(rgb_image)

    # 1. Draw Face Landmarks
    if face_result and face_result.face_landmarks:
        for face_landmarks in face_result.face_landmarks:
            drawing_utils.draw_landmarks(
                image=annotated_image,
                landmark_list=face_landmarks,
                connections=vision.FaceLandmarksConnections.FACE_LANDMARKS_TESSELATION,
                landmark_drawing_spec=None,
                connection_drawing_spec=drawing_styles.get_default_face_mesh_tesselation_style())
            drawing_utils.draw_landmarks(
                image=annotated_image,
                landmark_list=face_landmarks,
                connections=vision.FaceLandmarksConnections.FACE_LANDMARKS_CONTOURS,
                landmark_drawing_spec=None,
                connection_drawing_spec=drawing_styles.get_default_face_mesh_contours_style())

    # 2. Draw Pose Landmarks
    if pose_result and pose_result.pose_landmarks:
        for pose_landmarks in pose_result.pose_landmarks:
            drawing_utils.draw_landmarks(
                image=annotated_image,
                landmark_list=pose_landmarks,
                connections=vision.PoseLandmarksConnections.POSE_LANDMARKS,
                landmark_drawing_spec=drawing_styles.get_default_pose_landmarks_style())

    # 3. Draw Hand Landmarks
    if hand_result and hand_result.hand_landmarks:
        for hand_landmarks in hand_result.hand_landmarks:
            drawing_utils.draw_landmarks(
                image=annotated_image,
                landmark_list=hand_landmarks,
                connections=vision.HandLandmarksConnections.HAND_CONNECTIONS,
                landmark_drawing_spec=drawing_styles.get_default_hand_landmarks_style(),
                connection_drawing_spec=drawing_styles.get_default_hand_connections_style())

    return annotated_image

In [6]:
# Configuration and Initialization
BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = vision.FaceLandmarker
PoseLandmarker = vision.PoseLandmarker
HandLandmarker = vision.HandLandmarker

# Create landmarker instances
face_options = vision.FaceLandmarkerOptions(base_options=BaseOptions(model_asset_path='face_landmarker.task'))
pose_options = vision.PoseLandmarkerOptions(base_options=BaseOptions(model_asset_path='pose_landmarker.task'))
hand_options = vision.HandLandmarkerOptions(base_options=BaseOptions(model_asset_path='hand_landmarker.task'), num_hands=2)

video_path = '-7233869658737821499_clip0002.mp4'  # Replace with your video path
output_path = 'combined_output.mp4'

cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

with FaceLandmarker.create_from_options(face_options) as face_det, \
     PoseLandmarker.create_from_options(pose_options) as pose_det, \
     HandLandmarker.create_from_options(hand_options) as hand_det:

    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break

        # Convert to RGB for MediaPipe
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)

        # Run detection for all three tasks
        face_result = face_det.detect(mp_image)
        pose_result = pose_det.detect(mp_image)
        hand_result = hand_det.detect(mp_image)

        # Overlay all landmarks
        annotated_frame = draw_all_landmarks(rgb_frame, face_result, pose_result, hand_result)

        # Convert back to BGR for saving
        final_frame = cv2.cvtColor(annotated_frame, cv2.COLOR_RGB2BGR)
        out.write(final_frame)

cap.release()
out.release()
print(f"Processing complete. Result saved to: {output_path}")

Processing complete. Result saved to: combined_output.mp4
